In [1]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    dbname="ProjetoMC536",
    user="postgres",
    password="GSW30_curry",
    host="localhost",
    port="5432"
)

cursor = conn.cursor()

In [3]:
df = pd.read_csv('../datasets/DesmatamentoAreasIndigena.csv', sep=';')

df = df.rename(columns={
    'year': 'ano',
    'area km²': 'area_km2',
    'indi': 'area_indigena',
})

print(df.columns)
df['area_km2'] = df['area_km2'].str.replace(',', '.')

# for _, row in df.iterrows():
#     print(row)


Index(['ano', 'area_km2', 'area_indigena'], dtype='object')


In [8]:
# Armazena dados prontos para inserir depois
dados_desmatamento = []

for _, row in df.iterrows():
    ano = row['ano']
    area_km2 = row['area_km2']
    area_indigena = row['area_indigena']

    # Pega o id da area indígena
    cursor.execute("""
        SELECT id_area_indigena FROM public."AreaIndigena"
        WHERE nome = %s
    """, (area_indigena,))
    
    result = cursor.fetchone()
    id_area_indigena = result[0] if result else None
        
    dados_desmatamento.append((ano, area_km2, id_area_indigena, None))

In [9]:
from psycopg2.extras import execute_values

# Agora, insere **em lote**:
query = """
    INSERT INTO public."RelatorioDesmatamento" 
    (ano, area_km2, id_area_indigena, id_cidade)
    VALUES %s
"""
execute_values(cursor, query, dados_desmatamento)

# Finaliza
conn.commit()
cursor.close()
conn.close()

In [7]:
#Caso comando dê erro, desfaz as alterações
conn.rollback()